## Train / Validation Split — Embedded Interactions

This notebook assembles the final flat interaction table (with embeddings attached) and
splits it into train and validation sets for downstream Knowledge Tracing experiments.

**Pipeline:**
1. Load `maths_data_filtered.parquet` (36 837 students, 6 656 038 attempts).
2. Filter out interactions whose exercise has no compressed screenshot, leaving **6 263 671 interactions**.
3. Map string `user_id` values to contiguous integer indices (`user_id_int`) and assign a globally-unique `interaction_id` row index.
4. Reload the precomputed interaction embeddings from the 7 `interactions_embedded_part_*.parquet` files produced by `create_interactions_embedding.ipynb` and join them onto the attempt log.
5. Retain only the columns needed for modelling: `user_id_int`, `exercise_id`, `created_at`, `data_correct`, `interaction_embedding` (float32 array of size 576).
6. Split users 80 / 20 (seed 42) and write two output files to `data/`:
   - `train_interactions_embedded.parquet` — **5 000 085** interactions
   - `val_interactions_embedded.parquet` — **1 263 586** interactions

**Output:** flat, embedding-enriched attempt tables ready to be consumed by sequence-modelling or Knowledge Tracing training loops.


In [9]:

import polars as pl
import json
import pathlib
import os
import numpy as np
from exercise_embedding import encode_all
from attempt_embedding import encode_interactions
from tqdm import tqdm


DATASET = pathlib.Path("../../MIAAM")
EMBEDDING_FOLDER = pathlib.Path("../data")
SAVE_FOLDER = pathlib.Path("../data")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")

### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [10]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}
interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))


In [11]:
user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)
interactions = interactions.join(user_id_map, on="user_id", how="left")
del user_id_map

interactions = interactions.with_row_index("interaction_id")

In [12]:
part_paths = sorted(EMBEDDING_FOLDER.glob("interactions_embedded_part_*.parquet"))
print(f"Found {len(part_paths)} parts.")

interactions_embeddings = pl.concat([pl.read_parquet(p) for p in part_paths])
print(interactions_embeddings.shape)

Found 7 parts.
(6263671, 2)


In [13]:
# Assign stable row indices matching the interaction_id used by encode_interactions.
# Left joins above preserve row order, so this index is consistent with interactions_embeddings.

# Join exercise and interaction embeddings onto the attempt log
interactions = (
    interactions
    .join(
        interactions_embeddings.rename({"embedding": "interaction_embedding"}),
        on="interaction_id",
        how="left",
    )
)

In [14]:
interactions = interactions.select([
    "user_id_int", "exercise_id", "created_at", "data_correct", "interaction_embedding"
])

In [ ]:
# Split by user_id_int: 80% train, 20% val
user_ids = interactions.get_column("user_id_int").unique().to_list()

rng = np.random.default_rng(42)
rng.shuffle(user_ids)

split = int(len(user_ids) * 0.8)
train_ids = set(user_ids[:split])
val_ids = set(user_ids[split:])

train_df = interactions.filter(pl.col("user_id_int").is_in(train_ids))
val_df = interactions.filter(pl.col("user_id_int").is_in(val_ids))

del interactions



In [ ]:

train_df.write_parquet(SAVE_FOLDER / "train_interactions_embedded.parquet")
val_df.write_parquet(SAVE_FOLDER / "val_interactions_embedded.parquet")